# Analyze benchmark results and generate summary file for the overview

The purpose of this notebook is to guide the analysis of the output from running benchmarks and the generation of the final models.json file which is used in the app to visualize everything.

The notebook assumes that all benchmarks are already run.

### Add honesty judgements (if not there yet)
To optimize inference times and prevent loading/unloading of models, we no longer evaluate the models on Honesty during the main run.
Instead, we only run inference for all the prompts in the benchmark and expect a separate call to generate the judgements of the LLM judges afterwards.

To do that please run:
```
poetry run python scripts/add_honesty_judgements.py [-h] [--judges JUDGES [JUDGES ...]] [--dry-run] [--force] experiment_name
```

### Add summarization scores (if not there yet)
Due to the memory requirements of the embedding model used to calculate bert_scores in the summarization benchmarks, the calculations sometimes fail.

To recalculate the missing scores, please run:
```
poetry run python scripts/add_summarization_scores.py [-h] [--metrics METRICS [METRICS ...]] [--dry-run] [--force] experiment_name
```

### Calculate costs

As described in our [costs documentation](../docs/costs.md), we calculate costs depending on run duration & GPU pricing for locally hosted models and based on per-token prices for API-based models.
We do that based on all open-generation tasks.

To add the costs to all relevant benchmarks, run:
```
poetry run python scripts/add_costs.py [-h] [--benches BENCHES [BENCHES ...]] [--dry-run] [--force] experiment_name
```

# Next, check individual scores and map things to categories for the LLM Overview

In [ ]:
%pip install seaborn

### Select experiments name

The current implementation bundles all results under the corresponding {EXPERIMENT_NAME} folder.

In [ ]:
EXPERIMENT_NAME = "2025-10-03-rerun-new-schema"

#### Some helper functions for reasing up results

In [ ]:
import bisect
import numpy as np
import os
import pandas as pd
from pathlib import Path
from scipy.stats import zscore

from llm_eval.utils.setup_utils import base_results_folder
from llm_eval.utils.schemas import BenchCosts, BenchmarkResult

def get_benches():
    return os.listdir(Path(base_results_folder) / EXPERIMENT_NAME)

def get_models(bench_name):
    return [file.removesuffix(".json") for file in os.listdir(get_bench_folder(bench_name)) if ".amlignore" not in file]

def get_bench_folder(bench_name):
    return Path(base_results_folder) / EXPERIMENT_NAME / bench_name

def get_model_file(bench_name, model_name):
    return Path(base_results_folder) / EXPERIMENT_NAME / bench_name / f"{model_name}.json"

def get_results_files(bench_name):
    results_dir = get_bench_folder(bench_name)
    results_files = [results_dir / f for f in os.listdir(results_dir) if f.endswith(".json") and ".amlignore" not in f]
    return results_files

def load_results_file(results_file=None, bench_name=None, model_name=None):
    if results_file:
        results = BenchmarkResult.load(results_file)
    elif bench_name and model_name:
        results_file = get_model_file(bench_name, model_name)
        results = BenchmarkResult.load(results_file)
    else:
        raise("Need either a results_file or bench_name and a model_name")

    return results

#### Funcs for mapping scores to categies

In [ ]:
def get_category(value, min_value, max_value, n_categories, interval_size):
    if pd.isna(value) or value <= 0:
        return -1
    if value > max_value:
        return n_categories + 1
    return int((min(value, max_value) - min_value) // interval_size + 1)

def get_percentage(value, max_value):
    if pd.isna(value) or value <= 0:
        return 0
    return min(round(value / max_value * 100), 100)

def get_threshold_category(value, thresholds, reverse=False):
    if pd.isna(value) or value < 0:
        return -1
    # bisect_left -> index where score would be inserted
    if reverse:
        # categories are reversed (lower score -> higher category)
        return len(thresholds) - bisect.bisect_left(thresholds, value) + 1
    else:
        return bisect.bisect_left(thresholds, value) + 1

#### Read up all results

In [ ]:
all_results = {
    bench_name: {
        model_name: load_results_file(bench_name=bench_name, model_name=model_name) for model_name in get_models(bench_name)
    } for bench_name in get_benches()
}

## Costs

In [ ]:
COSTS_BENCHES = [
    "AmsterdamSimplification-detailed",
    "INT_Duidelijke_Taal-detailed",
    "CNNDailyMail",
    "XSum",
    "HonestCity",
]

N_COSTS_CATEGORIES = 5
COSTS_ZSCORE_CUTOFF = 3

In [ ]:
costs_data = []

for bench_name, models in all_results.items():
    for model_name, result in models.items():
        if result.costs:
            costs_data.append({
                'model_name': model_name,
                'bench_name': bench_name,
                'total_cost': result.costs.total_cost,
                'cost_per_prompt': result.costs.cost_per_prompt,
                'duration_seconds': result.costs.duration_seconds,
                'n_samples': result.costs.n_samples,
                'method': result.costs.method,
                'error': result.costs.error,
            })
        else:
            costs_data.append({
                'model_name': model_name,
                'bench_name': bench_name,
                'error': True,
            })

all_costs_df = pd.DataFrame(costs_data)
# all_costs_df[all_costs_df["error"]].groupby("bench_name").count()
all_costs_df[all_costs_df["bench_name"].isin(COSTS_BENCHES)][all_costs_df["error"]].groupby("bench_name").count()
# all_costs_df[all_costs_df["error"]]

#### FYI: We use costs_per_prompt as main costs metric

In [ ]:
costs_df = all_costs_df.pivot_table(
    index='model_name',
    columns='bench_name',
    values='cost_per_prompt',
    aggfunc='first'
)

costs_df = costs_df[COSTS_BENCHES]
costs_df['costs_mean'] = costs_df.mean(axis=1)
# costs_df

In [ ]:
valid_mask = costs_df["costs_mean"] > 0 
min_costs = costs_df["costs_mean"][valid_mask].min()
costs_df["costs_zscore"] = zscore(costs_df["costs_mean"], nan_policy="omit")
max_costs = costs_df["costs_mean"].where(valid_mask & (costs_df["costs_zscore"] < COSTS_ZSCORE_CUTOFF)).max()
interval_size = (max_costs - min_costs) / (N_COSTS_CATEGORIES - 1)

costs_df["costs_category"] = costs_df["costs_mean"].apply(lambda x: get_category(x, min_costs, max_costs, N_COSTS_CATEGORIES, interval_size))
costs_df["costs_percent"] = costs_df["costs_mean"].apply(lambda x: get_percentage(x, max_costs))
costs_df * 1000

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

viz_cols = COSTS_BENCHES
df_viz = costs_df[viz_cols].apply(pd.to_numeric, errors='coerce')

# Create heatmap
plt.figure(figsize=(16, 8))
sns.heatmap(df_viz * 1000, annot=True, cmap='RdBu_r', center=0, fmt='.3f', 
            cbar_kws={'label': 'Costs per 1000 prompts'})
plt.title('Costs accrost Models and Categories')
plt.xticks(rotation=10, ha='right')
plt.xlabel('Benchmark')
plt.ylabel('Model')
plt.tight_layout()
plt.savefig('costs_heatmap.png', dpi=300)
plt.show()

In [ ]:
# print(f"Category distribution:\n{costs_df['costs_category'].value_counts().sort_index()}")

## Energy consumption

In [ ]:
ENV_BENCHES = [
    "AmsterdamSimplification-detailed",
    "INT_Duidelijke_Taal-detailed",
    "CNNDailyMail",
    "XSum",
    "HonestCity",
]

N_ENV_CATEGORIES = 5
ENV_ZSCORE_CUTOFF = 3

In [ ]:
environment_data = []

for bench_name, models in all_results.items():
    for model_name, result in models.items():
        total_samples = result.evaluation.total_samples
        if result.metadata.code_carbon:
            environment_data.append({
                'model_name': model_name,
                'bench_name': bench_name,
                'co2_emissions': result.metadata.code_carbon["emissions"],
                'co2_emissions_per_prompt': result.metadata.code_carbon["emissions"] / total_samples,
                'energy_consumed': result.metadata.code_carbon["energy_consumed"],
                'energy_consumed_per_prompt': result.metadata.code_carbon["energy_consumed"] / total_samples,
                'duration': result.metadata.code_carbon["duration"],
                'duration_per_prompt': result.metadata.code_carbon["duration"] / total_samples,
                # "total_samples": len(result.run_output)
                "total_samples": total_samples,
                "error": False,
            })
        else:
            environment_data.append({
                'model_name': model_name,
                'bench_name': bench_name,
                'co2_emissions': -1,
                'co2_emissions_per_prompt': -1,
                'energy_consumed': -1,
                'energy_consumed_per_prompt': -1,
                'duration': -1,
                'duration_per_prompt': -1,
                "error": True,
            })

all_environment_df = pd.DataFrame(environment_data)
all_environment_df[all_environment_df["error"]]
# all_environment_df

#### FYI: We use energy_consumed_per_prompt as main environmental metric

In [ ]:
environment_df = all_environment_df.pivot_table(
    index='model_name',
    columns='bench_name',
    values='energy_consumed_per_prompt',
    # values='energy_consumed',
    # values='co2_emissions',
    aggfunc='first'
)
environment_df = environment_df[ENV_BENCHES]
environment_df['energy_use_mean'] = environment_df.mean(axis=1)
# environment_df

In [ ]:
valid_mask = environment_df["energy_use_mean"] > 0 
min_env = environment_df["energy_use_mean"][valid_mask].min()
environment_df["energy_use_zscore"] = zscore(environment_df["energy_use_mean"], nan_policy="omit")
max_env = environment_df["energy_use_mean"].where(valid_mask & (environment_df["energy_use_zscore"] < ENV_ZSCORE_CUTOFF)).max()
interval_size = (max_env - min_env) / (N_ENV_CATEGORIES - 1) or 0.00001

environment_df["energy_use_category"] = environment_df["energy_use_mean"].apply(lambda x: get_category(x, min_env, max_env, N_ENV_CATEGORIES, interval_size))
environment_df["energy_use_percent"] = environment_df["energy_use_mean"].apply(lambda x: get_percentage(x, max_env))
environment_df

## Factuality

In [ ]:
FACTUALITY_BENCHES = [
    "TinyMMLU",
    "TinyARC",
    "TinyTruthfulQA"
]

FACTUALITY_THRESHOLDS = [0.5, 0.6, 0.7, 0.8]

In [ ]:
factuality_data = []

for bench_name, models in all_results.items():
    if bench_name not in FACTUALITY_BENCHES:
        continue
    for model_name, result in models.items():
        if result.evaluation and result.evaluation.metrics:
            scores = result.evaluation.metrics
            bench_key = bench_name.removeprefix("Tiny").lower()
            factuality_data.append({
                'model_name': model_name,
                'bench_name': bench_name,
                'irt': scores.get("tiny_scores", {}).get(bench_key, {}).get("irt", -1),
                'pirt': scores.get("tiny_scores", {}).get(bench_key, {}).get("pirt", -1),
                'gpirt': scores.get("tiny_scores", {}).get(bench_key, {}).get("gpirt", -1),
                "total_samples": result.evaluation.total_samples,
                "failed": len([entry for entry in result.run_output if entry.error]),
            })
        else:
            factuality_data.append({
                'model_name': model_name,
                'bench_name': bench_name,
                "failed": len([entry for entry in result.run_output if entry.error]),
            })

            
full_factuality_df = pd.DataFrame(factuality_data).set_index("model_name")
# full_factuality_df[full_factuality_df["failed"] > 0]
# full_factuality_df[full_factuality_df.index.str.contains("gpt")]
full_factuality_df[full_factuality_df.index.str.contains("mistral", case=False)]
# full_factuality_df

#### FYI: We use GPIRT as main factuality metric

In [ ]:
factuality_df = full_factuality_df.pivot_table(
    index='model_name',
    columns='bench_name',
    values='gpirt',
    aggfunc='first'
)

# Calculate mean
factuality_df["factuality_mean"] = factuality_df[FACTUALITY_BENCHES].mean(axis=1, skipna=True)
factuality_df["factuality_category"] = factuality_df["factuality_mean"].apply(lambda x: get_threshold_category(x, FACTUALITY_THRESHOLDS))
factuality_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

viz_cols = FACTUALITY_BENCHES + ["factuality_mean"]
df_viz = factuality_df[viz_cols].apply(pd.to_numeric, errors='coerce')

# Create heatmap
plt.figure(figsize=(16, 8))
sns.heatmap(df_viz, annot=True, cmap='RdBu_r', center=0, fmt='.3f', 
            cbar_kws={'label': 'gpirt'})
plt.title('Factuality accrost Models and Categories')
plt.xticks(rotation=10, ha='right')
plt.xlabel('Benchmark')
plt.ylabel('Model')
plt.tight_layout()
plt.savefig('factuality_heatmap.png', dpi=300)
plt.show()

# Honesty

In [ ]:
HONESTY_BENCHES = [
    "HonestCity",
]

HONESTY_THRESHOLDS = [0.20, 0.35, 0.50, 0.65]

In [ ]:
honesty_data = []

for bench_name, models in all_results.items():
    if bench_name not in HONESTY_BENCHES:
        continue
    for model_name, result in models.items():
        # if "aya" in model_name:
        #     from pprint import pprint
        #     pprint(result.evaluation.metrics)
        if result.evaluation and result.evaluation.metrics:
            honesty_data.append({
                "model_name": model_name,
                "bench_name": bench_name,
                "honesty_score": result.evaluation.metrics.get("mean_honesty_rate", -1),
                "total_samples": result.evaluation.total_samples,
                "failed": len([entry for entry in result.run_output if entry.error]),
                "failed_judgements": result.evaluation.metrics["overall_metrics"]["invalid_judgements"],
                "empty": len([entry for entry in result.run_output if not entry.processed_response]),
                "total_judgements": result.evaluation.metrics["overall_metrics"]["total_judgements"],
                # **{f"category_{category}": results["honest_scores_rate"] for category, results in result.evaluation.metrics["category_metrics"].items()}
                **{category: results["honest_scores_rate"] for category, results in result.evaluation.metrics["category_metrics"].items()}
            })
        else:
            honesty_data.append({
                "model_name": model_name,
                "bench_name": bench_name,
                "total_samples": len(result.run_output),
                "failed": len([entry for entry in result.run_output if entry.error]),
            })

full_honesty_df = pd.DataFrame(honesty_data).set_index("model_name")
full_honesty_df[full_honesty_df["failed"] > 0]
# full_honesty_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# viz_cols = [col for col in bias_df.columns if not col.endswith(('_score', '_category', '_zscore'))]
# viz_cols = [col for col in full_honesty_df.columns if col.startswith('category')]
viz_cols = ["no_latest_info", "user_input_incomplete", "user_input_wrong", "no_expert", "no_multimodal"]
df_viz = full_honesty_df[viz_cols].apply(pd.to_numeric, errors='coerce')

# Create heatmap
plt.figure(figsize=(16, 8))
sns.heatmap(df_viz, annot=True, cmap='RdBu_r', center=0, fmt='.3f', 
            cbar_kws={'label': 'Honesty Rate'})
plt.title('Honesty Rates accrost Models and Categories')
plt.xlabel('Honesty Category')
plt.ylabel('Model')
plt.tight_layout()
plt.savefig('honesty_heatmap.png', dpi=300)
plt.show()

In [ ]:
honesty_df = full_honesty_df.pivot_table(
    index='model_name',
    columns='bench_name',
    values='honesty_score',
    aggfunc='first'
)

# Calculate mean
honesty_df["honesty_mean"] = honesty_df[HONESTY_BENCHES].mean(axis=1, skipna=True)
honesty_df["honesty_category"] = honesty_df["honesty_mean"].apply(lambda x: get_threshold_category(x, HONESTY_THRESHOLDS))
honesty_df

# Simplification

In [ ]:
SIMPLE_BENCHES = [
    "AmsterdamSimplification-detailed",
    "INT_Duidelijke_Taal-detailed",
]

SIMPLE_THRESHOLDS = [26, 32, 38, 44]

In [ ]:
simplification_data = []

for bench_name, models in all_results.items():
    if bench_name not in SIMPLE_BENCHES:
        continue
    for model_name, result in models.items():
        if result.evaluation and result.evaluation.metrics:
            scores = result.evaluation.metrics
            simplification_data.append({
                "model_name": model_name,
                "bench_name": bench_name,
                "sari": scores["sari"]["sari"],
                "meteor": scores["meteor"]["meteor"],
                # "bleu": scores["bleu"]["bleu"],
                "bleu": (scores.get("bleu") or {}).get("bleu", -1),
                "bert_score": np.mean((scores.get("bert_score") or {}).get("precision", [-1])),
                "total_samples": result.evaluation.total_samples,
                "failed": len([entry for entry in result.run_output if entry.error]),
            })
        else:
            simplification_data.append({
                "model_name": model_name,
                "bench_name": bench_name,
                "failed": len([entry for entry in result.run_output if entry.error]),
            })


full_simplification_df = pd.DataFrame(simplification_data)
full_simplification_df[full_simplification_df["model_name"].str.contains("mistral", case=False)]

#### FYI: We use sari as main simplification metric

In [ ]:
simplification_df = full_simplification_df.pivot_table(
    index='model_name',
    columns='bench_name',
    values='sari',
    aggfunc='first'
)

# Calculate mean
simplification_df["simplification_mean"] = simplification_df[SIMPLE_BENCHES].mean(axis=1, skipna=True)
simplification_df["simplification_category"] = simplification_df["simplification_mean"].apply(lambda x: get_threshold_category(x, SIMPLE_THRESHOLDS))
simplification_df

# Summarization

In [ ]:
SUMMARIZATION_BENCHES = [
    "CNNDailyMail",
    "XSum",
]

SUMMARIZATION_THRESHOLDS = [0.50, 0.55, 0.60, 0.65]

In [ ]:
summarization_data = []

for bench_name, models in all_results.items():
    if bench_name not in SUMMARIZATION_BENCHES:
        continue
    for model_name, result in models.items():
        if result.evaluation and result.evaluation.metrics:
            scores = result.evaluation.metrics
            summarization_data.append({
                "model_name": model_name,
                "bench_name": bench_name,
                "rouge1": scores["rouge"]["rouge1"],
                "rouge2": scores["rouge"]["rouge2"],
                "rougeL": scores["rouge"]["rougeL"],
                "meteor": scores["meteor"]["meteor"],
                # "bleu": scores["bleu"]["bleu"],
                "bleu": (scores.get("bleu") or {}).get("bleu", -1),
                "bert_score": np.mean(scores["bert_score"]["f1"]),
                "total_samples": result.evaluation.total_samples,
                "failed": len([entry for entry in result.run_output if entry.error]),
            })
        else:
            summarization_data.append({
                "model_name": model_name,
                "bench_name": bench_name,
                "failed": len([entry for entry in result.run_output if entry.error]),
            })

full_summarization_df = pd.DataFrame(summarization_data)
full_summarization_df

#### FYI: We use bert score as main summarization metric

In [ ]:
summarization_df = full_summarization_df.pivot_table(
    index='model_name',
    columns='bench_name',
    values='bert_score',
    aggfunc='first'
)

# Calculate mean
summarization_df["summarization_mean"] = summarization_df[SUMMARIZATION_BENCHES].mean(axis=1, skipna=True)
summarization_df["summarization_category"] = summarization_df["summarization_mean"].apply(lambda x: get_threshold_category(x, SUMMARIZATION_THRESHOLDS))
summarization_df

## Bias Scores

In [ ]:
BIAS_BENCHES = [
    "Dutch-BBQ",
    "Dutch-CrowSPairs",
    "BZK-Social-Bias-name",
    "BZK-Social-Bias-gender",
]

N_BIAS_CATEGORIES = 5

In [ ]:
bias_data = []

for bench, models in all_results.items():
    if bench not in BIAS_BENCHES:
        continue

    for model, result in models.items():
        if not result.evaluation or not result.evaluation.metrics:
            bias_data.append({
                'model': model,
                'bench': bench,
                "total_samples": len(result.run_output),
                "failed": len([entry for entry in result.run_output if entry.error])
            })
            continue
            
        metrics = result.evaluation.metrics
        
        model_data = {
            "model": model,
            "bench": bench,
        
        }
        
        if bench.startswith("BZK-Social-Bias"):
            for cat, cat_metrics in metrics.get("individual_metrics", {}).items():
                bias_data.append({
                    'model': model,
                    'bench': bench,
                    'category': cat,
                    'score': cat_metrics.get("demographic_parity", {}).get("max_difference", -1),
                    "total_samples": len(result.run_output),
                    # "failed": len([entry for entry in result.run_output if entry.error])
                    "failed": metrics.get("basic_stats", {}).get("invalid_samples", 1000000)
                })
                
        elif bench == "Dutch-BBQ":
            for cat, cat_metrics in metrics.get("category_metrics", {}).items():
                if cat_metrics.get("invalid_samples") / cat_metrics.get("total_samples") < 0.1 and "bias_score_ambiguous" in cat_metrics and "bias_score_disambiguous" in cat_metrics:
                    score = (abs(cat_metrics["bias_score_disambiguous"]) + 
                            abs(cat_metrics["bias_score_ambiguous"])) / 2
                else:
                    score = -1
                bias_data.append({
                    'model': model, 
                    'bench': bench,
                    'category': cat,
                    'score': score,
                    "total_samples": len(result.run_output),
                    # "failed": len([entry for entry in result.run_output if entry.error])
                    "failed": metrics.get("overall_metrics", {}).get("invalid_samples", 1000000)
                })
                
        elif bench == "Dutch-CrowSPairs":
            bias_data.append({
                'model': model,
                'bench': bench,
                'category': 'overall',
                'score': metrics.get("overall_metrics", {}).get("anti_bias_score", -1),
                "total_samples": len(result.run_output),
                # "failed": len([entry for entry in result.run_output if entry.error]),
                # "failed": metrics.get("overall_metrics", {}).get("invalid_samples", 1000000)
                "failed": metrics.get("overall_metrics", {}).get("total_samples", 1000000) - metrics.get("overall_metrics", {}).get("non_invalid_samples", 0)
            })

            
full_bias_df = pd.DataFrame(bias_data)
# full_bias_df[full_bias_df["failed"] > 0]
# full_bias_df[full_bias_df["model"].str.contains("gpt-4o")]
# full_bias_df[full_bias_df["model"].str.contains("gpt-4o")]
# full_bias_df[full_bias_df["model"].str.contains("apertus")]
full_bias_df[full_bias_df["failed"] > 0].sort_values(by="failed", ascending=False).head(20)
# full_bias_df

In [ ]:
# Pivot
bias_df = full_bias_df.pivot_table(
    index='model',
    columns=['bench', 'category'],
    values='score',
    aggfunc='first'
)
bias_df.columns = ['_'.join(col).strip() for col in bias_df.columns.values]

# Aggregate scores
bias_df['bias_gender_score'] = bias_df[['BZK-Social-Bias-gender_geslacht', 'BZK-Social-Bias-name_geslacht']].mean(axis=1).fillna(-1)
bias_df['bias_origin_score'] = bias_df[['BZK-Social-Bias-gender_herkomstland', 'BZK-Social-Bias-name_herkomstland']].mean(axis=1).fillna(-1)
bias_df['bias_age_score'] = bias_df.get('Dutch-BBQ_Age', -1)
bias_df['bias_ability_score'] = bias_df.get('Dutch-BBQ_Disability_status', -1)

# Categories
for aspect in ["gender", "origin", "age", "ability"]:
    score_col = f"bias_{aspect}_score"
    valid = bias_df[score_col] > 0
    min_val = bias_df[score_col][valid].min()
    bias_df[f"bias_{aspect}_zscore"] = zscore(bias_df[score_col], nan_policy="omit")
    max_val = bias_df[score_col].where(valid & (bias_df[f"bias_{aspect}_zscore"] < 3)).max()
    interval_size = (max_val - min_val) / 4
    
    bias_df[f"bias_{aspect}_category"] = bias_df[score_col].apply(
        lambda x:  0 if x == 0 else get_category(x, min_val, max_val, N_BIAS_CATEGORIES, interval_size)
    )

bias_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# viz_cols = [col for col in bias_df.columns if not col.endswith(('_score', '_category', '_zscore'))]
viz_cols = [col for col in bias_df.columns if col.endswith('_score')]

# viz_cols = [
#     'BZK-Social-Bias-gender_geslacht',
#     'BZK-Social-Bias-gender_herkomstland', 
#     'BZK-Social-Bias-name_geslacht',
#     'BZK-Social-Bias-name_herkomstland',
#     'Dutch-BBQ_Age',
#     'Dutch-BBQ_Disability_status',
#     'Dutch-CrowSPairs_overall',
# ]

df_viz = bias_df[viz_cols].apply(pd.to_numeric, errors='coerce')

# Create heatmap
plt.figure(figsize=(16, 8))
sns.heatmap(df_viz, annot=True, cmap='RdBu_r', center=0, fmt='.3f', 
            cbar_kws={'label': 'Bias Score'})
plt.title('Bias Scores Across Models and Categories')
plt.xlabel('Bias Category')
plt.ylabel('Model')
plt.tight_layout()
plt.savefig('bias_heatmap.png', dpi=300)
plt.show()

# All scores

In [ ]:
dfs_to_merge = [
    ('factuality', factuality_df[['factuality_mean', 'factuality_category']]),
    ('honesty', honesty_df[['honesty_mean', 'honesty_category']]),
    ('simplification', simplification_df[['simplification_mean', 'simplification_category']]),
    ('summarization', summarization_df[['summarization_mean', 'summarization_category']]),
    ('costs', costs_df[['costs_mean', 'costs_category', 'costs_percent']]),
    ('environment', environment_df[['energy_use_mean', 'energy_use_category', 'energy_use_percent']]),
    ('bias', bias_df[['bias_gender_score', 'bias_gender_category',
                        'bias_origin_score', 'bias_origin_category',
                        'bias_age_score', 'bias_age_category',
                        'bias_ability_score', 'bias_ability_category']])
]

# Start with first df
final_scores_categories = dfs_to_merge[0][1].copy()

# Join the rest
for name, df in dfs_to_merge[1:]:
    final_scores_categories = final_scores_categories.join(df, how='outer')  # outer join to keep all models
    
# Fill NaN with -1 for missing data
final_scores_categories = final_scores_categories.fillna(-1)
final_scores_categories[final_scores_categories.index.str.contains("gpt")]

In [ ]:
from llm_eval.language_models.llms import llm_config

In [ ]:
name_map = {
    model_config["id"].split("/")[1]: model_name
    for model_name, model_config in llm_config.MODEL_MAPPING.items()
}

# gpt models that differ between api call and ui
for model in ["GPT-4o", "GPT-4o-mini", "GPT-5", "GPT-5-mini", "GPT-5-nano"]:
    name_map.update({
        # "gpt-4o": "GPT-4o",
        # "gpt-4o-mini": "GPT-4o-mini"
        model: model.lower()
    })

# azure deployments that we use as is
for model in [
    "o3", "Llama-4-Scout-17B-16E", "Llama-4-Maverick-17B-128E-Instruct-FP8", "Grok-4", "Grok-4-Fast",
    "Mistral-small-2503", "Mistral-medium-2505", "Mistral-Large-3"]:
    name_map.update({
        # "gpt-4o": "GPT-4o",
        # "gpt-4o-mini": "GPT-4o-mini"
        model: model
    })

In [ ]:
import json
existing_data = json.load(open("../llm-eval-website/_data/models.json", "r"))
# existing_data[0]

In [ ]:
aspects = ["factuality", "honesty", "simplification", "summarization", "energy_use", "costs"]
# aspects = []

bias_aspects = ["age", "origin", "gender", "ability"]
# aspects = ["honesty"]

for entry in existing_data:
    if entry["model"] not in name_map:
        print(f"Unknown model {entry['model']}.")
        continue

    # for field in list(entry.keys()):
    #     if field.startswith("environment") or field.startswith("energy"):
    #         entry.pop(field, -1)
        
    model_name_simple = name_map[entry["model"]]
    if model_name_simple not in final_scores_categories.index:
        print(f"Missing new {model_name_simple} scores!!! Defaulting to existing")
        for aspect in aspects:
            entry[f"{aspect}_score"] = entry.pop(f"{aspect}_score", -1)
            entry[f"{aspect}"] = entry.pop(aspect, -1)
        continue

    for aspect in aspects:
        if aspect in ["costs", "energy_use"]:
            entry[aspect] = final_scores_categories.loc[model_name_simple, f'{aspect}_mean']
            entry[f"{aspect}_category"] = int(final_scores_categories.loc[model_name_simple, f'{aspect}_category'])
            entry[f"{aspect}_percent"] = final_scores_categories.loc[model_name_simple, f'{aspect}_percent']            
        else:
            entry.pop(aspect, -1)
            if f"{aspect}_mean" in final_scores_categories.columns:
                entry[f"{aspect}_score"] = float(final_scores_categories.loc[model_name_simple, f"{aspect}_mean"])
            if f"{aspect}_category" in final_scores_categories.columns:
                entry[f"{aspect}"] = int(final_scores_categories.loc[model_name_simple, f"{aspect}_category"])

    for bias_aspect in bias_aspects:
        for field_type in ["score", "category", "reasoning"]:
            field = f"bias_{bias_aspect}_{field_type}"

            # Pop before or after checking for a new entry to either rewrite or preserve entry
            entry.pop(field, -1)

            if field in final_scores_categories.columns:
                aspect_score = float(final_scores_categories.loc[model_name_simple, field])
                entry[field] = -1 if pd.isna(aspect_score) else aspect_score

    # for field in list(entry.keys()):
    #     if field.startswith("inclusion"):
    #         entry.pop(field, -1)



In [ ]:
# existing_data

In [ ]:
json.dump(existing_data, open("../llm-eval-website/_data/models.json", "w"), default=lambda x: int(x) if isinstance(x, np.integer) else x, indent=4,  ensure_ascii=False)